# Module 12: Structural Breaks and Changepoints

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

[Module 11](../Module_11_Interrupted_Time_Series.md) asked whether something
changed on a date you were given. This module asks the harder question: **did
something change on a date nobody told you about, and how would you know?**

That happens constantly in public safety data. A records system is replaced, a
category is redefined, a reporting requirement changes, a unit is disbanded.
None of it appears in a data dictionary, and all of it breaks a trend.

The module also covers the mistake that searching for a date makes almost
inevitable, and the arithmetic that fixes it.

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm

prov = set(monthly[monthly["provisional"] == 1]["year_month"])
bytype = pd.read_csv(BASE + "cfs_monthly_by_type.csv")
bytype = bytype[~bytype["year_month"].isin(prov)]


def wide(agency_id):
    w = (bytype[bytype["agency_id"] == agency_id]
         .pivot(index="year_month", columns="incident_type", values="n_calls")
         .sort_index())
    w.index = pd.PeriodIndex(w.index, freq="M").to_timestamp()
    return w


def design(index):
    """Intercept, linear trend, and one harmonic. Everything a break sits on top of."""
    n = len(index)
    return np.column_stack([np.ones(n), np.arange(n) / 12.0,
                            np.sin(2 * np.pi * index.month.values / 12),
                            np.cos(2 * np.pi * index.month.values / 12)])


def scan(y, X, trim=0.15):
    """Fit a level shift at every candidate date and return the Wald statistic."""
    n = len(y)
    k0 = int(trim * n)
    rows = []
    for k in range(k0, n - k0):
        d = (np.arange(n) >= k).astype(float)
        r = sm.OLS(y, np.column_stack([X, d])).fit()
        rows.append((k, float((r.params[-1] / r.bse[-1]) ** 2), float(r.params[-1])))
    return rows


a = wide("A003")                       # Havenbrook
X = design(a.index)
print(f"{len(a)} months, {a.shape[1]} incident types")

## 2. The Chow test, when the date is known

If somebody tells you the date, the test is straightforward: fit the model with
and without a level shift at that date and compare.

In [ ]:
y = np.log(a["Public Order Offense"].values.astype(float))
d = (a.index >= "2023-01-01").astype(float)

r0 = sm.OLS(y, X).fit()
r1 = sm.OLS(y, np.column_stack([X, d])).fit()
F = ((r0.ssr - r1.ssr) / 1) / (r1.ssr / r1.df_resid)

from scipy import stats
print(f"  Havenbrook public order offences, break tested at 2023-01")
print(f"    shift {100 * (np.exp(r1.params[-1]) - 1):+.1f} percent")
print(f"    F = {F:.1f},  p = {1 - stats.f.cdf(F, 1, r1.df_resid):.2e}")

Unambiguous. But this only works when someone tells you the date, and the
p value is only valid because the date came from outside the data.

## 3. When nobody tells you the date

Fit a level shift at every candidate date and take the largest statistic. The
first and last 15 percent are excluded, because a break in the first few months
is indistinguishable from a different intercept.

In [ ]:
rows = scan(y, X)
k, w, b = max(rows, key=lambda z: z[1])
print(f"  strongest candidate: {str(a.index[k])[:7]}")
print(f"    Wald statistic {w:.1f}, shift {100 * (np.exp(b) - 1):+.1f} percent")
print(f"\n  the date the generator actually used: 2023-01")

Found, to the month.

Now the same scan on the **total** calls for service, where the reclassification
moved calls between two categories and left the sum alone.

In [ ]:
tot = np.log(a.sum(axis=1).values.astype(float))
k2, w2, b2 = max(scan(tot, X), key=lambda z: z[1])
oth = np.log(a["Other"].values.astype(float))
k3, w3, b3 = max(scan(oth, X), key=lambda z: z[1])

for name, k_, w_, b_ in [("public order offences", k, w, b),
                         ("Other", k3, w3, b3),
                         ("all calls, the total", k2, w2, b2)]:
    print(f"  {name:22s} best date {str(a.index[k_])[:7]}   Wald {w_:7.2f}   "
          f"shift {100 * (np.exp(b_) - 1):+6.1f}%")

Two components break hard in the same month, in opposite directions, and the
total shows nothing worth a second look.

**A dashboard built on totals would never have seen this.** Any trend,
forecast or peer comparison involving Havenbrook's public order category across
January 2023 is comparing two different definitions of the category.

## 4. The mistake that searching makes

The scan above tested 62 candidate dates and reported the largest statistic.
Comparing that maximum against the critical value for a **single** test is
wrong, and it is wrong by a lot.

Rather than assert that, simulate it: 400 series with a trend, a season, noise,
and **no break anywhere.**

In [ ]:
rng = np.random.default_rng(7)
n = 88
idx = pd.period_range("2019-01", periods=n, freq="M").to_timestamp()
Xs = design(idx)
beta = np.array([3.0, -0.05, 0.1, 0.15])

maxes = np.array([max(v for _, v, _ in scan(Xs @ beta + rng.normal(0, 0.25, n), Xs))
                  for _ in range(400)])

print("  400 series, no break in any of them. The largest Wald found by searching:")
print(f"    median {np.median(maxes):.1f}")
print(f"    exceeds 3.84, the chi square 5 percent value:  "
      f"{100 * (maxes > 3.84).mean():.0f}% of the time")
print(f"    exceeds 8.85, the Andrews 5 percent value:     "
      f"{100 * (maxes > 8.85).mean():.0f}% of the time")

**Sixty two percent.** Use the ordinary critical value after searching for the
date and you will find a significant break in most series that do not have one.

The correct reference distribution is the one for the **maximum** over
candidate dates, tabulated by Andrews. For one shifting parameter with 15
percent trimming the 5 percent value is **8.85**, not 3.84, and the 1 percent
value is 12.35.

The simulation clears 8.85 about 9 percent of the time rather than 5, which is
worth stating plainly: with 88 months and four nuisance parameters the
asymptotic critical value is still a little optimistic. Against Havenbrook's
Wald of 254 the distinction does not matter. Against a Wald of 9 it decides the
answer.

| After searching for the date | Use |
|---|---|
| 1 shifting parameter, 15 percent trim | 7.12 at 10 percent, 8.85 at 5, 12.35 at 1 |
| A statistic near the threshold | a simulation like the one above, on your own design |
| A date supplied from outside the data | the ordinary chi square value, and say where the date came from |

## 5. Three things that are not breaks

A level shift model, pointed at a series, will always return a best candidate
date. Most of the time it means nothing.

In [ ]:
def best_break(agency_id, column="rate"):
    g = final[final["agency_id"] == agency_id].sort_values("year_month")
    i = pd.PeriodIndex(g["year_month"], freq="M").to_timestamp()
    v = np.log((g["n_uof"] / g["n_arrests"]).values.astype(float))
    Xg = design(i)
    k_, w_, b_ = max(scan(v, Xg), key=lambda z: z[1])
    return str(i[k_])[:7], w_, 100 * (np.exp(b_) - 1)

for aid, why in [("A002", "one extreme month, June 2021"),
                 ("A007", "a steeper trend for the whole period"),
                 ("A008", "nothing planted at all")]:
    date, w_, sh = best_break(aid)
    verdict = "ABOVE 8.85" if w_ > 8.85 else "below 8.85"
    print(f"  {aid}  {why:38s} best {date}  Wald {w_:5.2f}  "
          f"{sh:+6.1f}%   {verdict}")

None of them clears the threshold, which is the correct answer in all three
cases. An outlier is not a break. A different slope is not a level break. And
a series with nothing in it still produces a best candidate date.

Note what Tarnbridge's best candidate is: **2023-11**, the month the training
programme was fully in place. The break is real and the test cannot see it,
which is the same conclusion [Module 6](../Module_06_Regression_With_ARMA_Errors.md)
reached from a different direction. A 12 percent step at one agency is below
what a single series can resolve.

## 6. How certain is the date itself

A significant break does not come with a certain date. Invert the likelihood
to get the set of dates the data cannot rule out.

In [ ]:
rows = scan(y, X)
lls = []
for k_, _, _ in rows:
    dd = (np.arange(len(y)) >= k_).astype(float)
    lls.append((k_, sm.OLS(y, np.column_stack([X, dd])).fit().llf))
top = max(v for _, v in lls)
inside = [k_ for k_, v in lls if 2 * (top - v) <= 3.84]
print(f"  dates the data cannot rule out: "
      f"{str(a.index[min(inside)])[:7]} to {str(a.index[max(inside)])[:7]}")
print(f"  that is {len(inside)} month(s) out of {len(rows)} tested")

A single month, because a 51 percent shift in a series this regular leaves no
room for doubt. **Do not expect that.** A break of 10 percent in a noisy series
routinely produces a plausible set spanning a year, and reporting the argmax
as "the date the policy took effect" then overstates what you know.

Always report the set, not just the peak.

## 7. A practical order of operations

| Step | Why |
|---|---|
| Ask the agency first | most breaks have a documented cause and a known date |
| Plot the components, not only the total | section 3 |
| Fix the date from outside the data if you can | it restores the ordinary critical value |
| If you search, use the searched critical value | section 4 |
| Check that it is not one outlier | section 5 |
| Check that it is not a slope difference | section 5 |
| Report the plausible date range | section 6 |
| Then decide what to do about it | split the series, add an indicator, or drop the affected span |

A break you can explain is a data quality note. A break you cannot explain is a
reason to call whoever maintains the records system, not a finding.

## Exercise

Havenbrook's break was found in the log of the level. Would a scan of the
**share** of calls in each category have found it too, and would it be easier
to interpret?

In [ ]:
# Fill in the blank, then run.
USE_SHARES = None          # try True

if USE_SHARES is not None:
    frame = a.div(a.sum(axis=1), axis=0) if USE_SHARES else a
    label = "share of all calls" if USE_SHARES else "count of calls"
    print(f"  scanning the {label}\n")
    for col in ["Public Order Offense", "Other", "Vehicle Stop"]:
        v = np.log(frame[col].values.astype(float))
        k_, w_, b_ = max(scan(v, X), key=lambda z: z[1])
        print(f"    {col:22s} {str(a.index[k_])[:7]}   Wald {w_:7.1f}   "
              f"shift {100 * (np.exp(b_) - 1):+6.1f}%")
else:
    print("Set USE_SHARES above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

Run it both ways.

```python
USE_SHARES = True     # then False
```

Both find January 2023 in the two affected categories, and the shares find it
more sharply: the Wald statistic rises from 254 to 279 for public order and
from 132 to 212 for Other. Dividing by the total removes the common growth the
trend term was otherwise absorbing.

Now look at the third row. Vehicle stops did not change at all, and on the
count scale their best candidate carries a Wald of 6.2. On the share scale it
rises to **7.3**, still under the 8.85 threshold but visibly closer to it.

That movement is not noise, it is arithmetic. **Shares are constrained to sum
to one.** When one category's share jumps 50 percent, every other category's
share is mechanically pushed the other way, whether or not anything happened
to it. Here the reclassification was small enough relative to the total that
the induced shift stays below the threshold. A larger one would not.

So the trade is real even though it did not bite this time. Shares are easier
to read and they push innocent categories toward significance. Counts are
noisier and they implicate only the categories that moved.

**Scan the counts to find out what changed. Report the shares to explain what
it means.** Never scan shares and then name every category that lit up.

</details>

---

**Next:** [Module 13: Intervention Analysis and Transfer Functions](Module_13_Intervention_Analysis.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*